In [4]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from sklearn.metrics.pairwise import(
    cosine_similarity,
    euclidean_distances
)

load_dotenv()

client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

EMBEDDING_MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2:free"
print(EMBEDDING_MODEL)



nvidia/llama-nemotron-embed-vl-1b-v2:free


In [5]:
documents = [
    {"id": 1, "text": "Spring Boot simplifies building production-ready Java backend applications.", "topic": "backend", "level": "beginner"},
    {"id": 2, "text": "Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.", "topic": "devops", "level": "intermediate"},
    {"id": 3, "text": "Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.", "topic": "databases", "level": "beginner"},
    {"id": 4, "text": "Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.", "topic": "messaging", "level": "intermediate"},
    {"id": 5, "text": "PostgreSQL is a relational database with strong support for transactions, indexing, and extensibility.", "topic": "databases", "level": "beginner"},
    {"id": 6, "text": "Vector databases store embeddings and make similarity search over high-dimensional vectors efficient.", "topic": "ai", "level": "beginner"},
    {"id": 7, "text": "RAG retrieves relevant external context and gives that context to a language model before generating an answer.", "topic": "ai", "level": "beginner"},
    {"id": 8, "text": "An embedding model converts text into a dense numerical vector that captures semantic meaning.", "topic": "ai", "level": "beginner"},
    {"id": 9, "text": "Cosine similarity compares the angle between vectors and is commonly used for semantic text retrieval.", "topic": "ai", "level": "intermediate"},
    {"id": 10, "text": "HNSW builds a navigable graph so approximate nearest-neighbor search can avoid scanning every vector.", "topic": "ai", "level": "advanced"},
    {"id": 11, "text": "Exact nearest-neighbor search compares the query against every candidate vector to obtain true nearest results.", "topic": "ai", "level": "intermediate"},
    {"id": 12, "text": "Quantization compresses vector representations so a vector database can reduce memory usage and improve search efficiency.", "topic": "ai", "level": "advanced"},
    {"id": 13, "text": "Payload indexes allow a vector database to combine semantic similarity search with structured filters.", "topic": "databases", "level": "intermediate"},
    {"id": 14, "text": "Python is widely used for machine learning, data processing, and building AI applications.", "topic": "programming", "level": "beginner"},
    {"id": 15, "text": "Async programming can improve throughput for applications that handle many independent I/O operations.", "topic": "programming", "level": "intermediate"},
    {"id": 16, "text": "API gateways can provide authentication, rate limiting, and request routing for microservices.", "topic": "backend", "level": "intermediate"},
    {"id": 17, "text": "Microservices split an application into independently deployable components that communicate over APIs.", "topic": "architecture", "level": "intermediate"},
    {"id": 18, "text": "Observability combines logs, metrics, and traces to understand the behavior of distributed systems.", "topic": "devops", "level": "intermediate"},
    {"id": 19, "text": "A cache can reduce database load and latency when frequently requested data can be reused safely.", "topic": "databases", "level": "beginner"},
    {"id": 20, "text": "Reranking can improve retrieval quality by applying a more expensive relevance model to an initial candidate set.", "topic": "ai", "level": "advanced"},
]

texts = [d["text"] for d in documents]

print(f"Total documents: {len(documents)}")

Total documents: 20


In [6]:
def embed_documents(texts: list[str])-> list[list[float]]:
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [d.embedding for d in resp.data]

doc_embeddings = embed_documents(texts)
print("Number of embeddings: ", len(doc_embeddings))
print("Dimensions per vector: ", len(doc_embeddings[0]))
print("first 5 values:", doc_embeddings[0][:5])

Number of embeddings:  20
Dimensions per vector:  2048
first 5 values: [0.0078887939453125, 0.02349853515625, 0.0030727386474609375, 0.013763427734375, 0.0362548828125]


In [7]:
query = "which programming language is commonly used for AI applications?"

query_embedding = embed_documents([query])  # returns a list with one vector

print("query:", query)
print("dimensions:", len(query_embedding[0]))
print("collectin must use size = ", len(doc_embeddings[0]))

query: which programming language is commonly used for AI applications?
dimensions: 2048
collectin must use size =  2048


In [8]:
cos = cosine_similarity(query_embedding, doc_embeddings)[0]
l2 = euclidean_distances(query_embedding, doc_embeddings)[0]

print(f"{'cos':>6} {'L2':>5}       Document")
print("-" * 70)
for doc , c, d in zip(documents, cos, l2):
    print(f"{c:6.3f} {d:6.3f} {doc}")

   cos    L2       Document
----------------------------------------------------------------------
 0.158  1.297 {'id': 1, 'text': 'Spring Boot simplifies building production-ready Java backend applications.', 'topic': 'backend', 'level': 'beginner'}
 0.093  1.346 {'id': 2, 'text': 'Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.', 'topic': 'devops', 'level': 'intermediate'}
 0.206  1.260 {'id': 3, 'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'}
 0.184  1.278 {'id': 4, 'text': 'Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.', 'topic': 'messaging', 'level': 'intermediate'}
 0.078  1.357 {'id': 5, 'text': 'PostgreSQL is a relational database with strong support for transactions, indexing, and extensibility.', 'topic': 'databases', 'level': 'beginner'}
 0.142  1.310 {'id': 6, 'text': 'Vector data

In [10]:
def retrieve(query_emb, doc_embs, docs, top_k = 2):
    scores = cosine_similarity(query_emb, doc_embs)[0]

    #pair each document with its score, sort highest first, take top_k
    scored = list(zip(docs, scores))
    print(scored)
    scored.sort(key = lambda x: x[1], reverse=True)

    return [{"document": doc, "score": float(score)} for doc, score in scored[:top_k]]

print("Top-k results:\n")
for hit in retrieve(query_embedding, doc_embeddings, documents, top_k=4):
    print(f"{hit['score']:.3f} {hit['document']}")

Top-k results:

[({'id': 1, 'text': 'Spring Boot simplifies building production-ready Java backend applications.', 'topic': 'backend', 'level': 'beginner'}, np.float64(0.15845897747885027)), ({'id': 2, 'text': 'Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.', 'topic': 'devops', 'level': 'intermediate'}, np.float64(0.09344970804806699)), ({'id': 3, 'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'}, np.float64(0.20588064470433287)), ({'id': 4, 'text': 'Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.', 'topic': 'messaging', 'level': 'intermediate'}, np.float64(0.18355448663961307)), ({'id': 5, 'text': 'PostgreSQL is a relational database with strong support for transactions, indexing, and extensibility.', 'topic': 'databases', 'level': 'beginner'}, np.float64(0.07816842303505184)), ({'id': 6, 'text

#####Qdrant Vector Database


In [11]:
from qdrant_client import QdrantClient, models

# qdrant = QdrantClient(":memory:")

qdrant = QdrantClient(url = "http://localhost:6333")


In [12]:
COLLECTION = "rag_fundamentals"

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(
        size = len(doc_embeddings[0]),
        distance = models.Distance.COSINE
    )
)

print("created:", COLLECTION, "| size:", len(doc_embeddings[0]))

created: rag_fundamentals | size: 2048


In [13]:
points = [
    models.PointStruct(
        id = doc["id"],
        vector = vec,
        payload = {"text": doc["text"], "topic": doc["topic"], "level": doc["level"]},
    )
    for doc, vec in zip(documents, doc_embeddings)
]

qdrant.upsert(collection_name=COLLECTION, points=points, wait = True)
print(f"Inserted {len(points)} points.")

Inserted 20 points.


In [14]:
results = qdrant.query_points(
    collection_name=COLLECTION,
    query=query_embedding[0],
    limit=5,
    with_payload=True
).points

for p in results:
    print(p)

id=14 version=1 score=0.6389383 payload={'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=15 version=1 score=0.28941315 payload={'text': 'Async programming can improve throughput for applications that handle many independent I/O operations.', 'topic': 'programming', 'level': 'intermediate'} vector=None shard_key=None order_value=None
id=17 version=1 score=0.2159392 payload={'text': 'Microservices split an application into independently deployable components that communicate over APIs.', 'topic': 'architecture', 'level': 'intermediate'} vector=None shard_key=None order_value=None
id=3 version=1 score=0.20588064 payload={'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=4 version=1 score=0.18355452 payload={'